# Ghép hai nguồn để chọn thị trường: doanh số nội bộ + World Bank

**Bối cảnh.** Tháng 12/2011, Sales Director của cửa hàng quà tặng online (Online Retail II) hỏi: *"Ngoài Anh, năm 2012
nên dồn ngân sách marketing vào nước châu Âu nào?"* Doanh số theo nước có sẵn trong dữ liệu nội bộ, nhưng "nước bán
nhiều" chưa chắc là "nước còn dư địa". Bạn ghép thêm dân số và GDP bình quân đầu người từ World Bank API để chuẩn hoá
theo quy mô thị trường.

Việc khó không nằm ở API mà ở **chỗ nối**: dữ liệu nội bộ ghi `EIRE`, `RSA`, `USA`, `Czech Republic`, `European
Community`, `Unspecified`; World Bank ghi `Ireland`, `South Africa`, `United States`, `Czechia`. Nối sai hoặc bỏ sót thì
một phần doanh thu biến mất khỏi bảng mà không ai biết. Và doanh thu một nước nhỏ có thể chỉ là một hai khách sỉ.

**Cách làm bài**
- Hai hàm được chấm (`match_countries`, `coverage`); phần còn lại là phân tích của bạn trong các cell `explore` và
  `MEMO.md`.
- Cell `explore` đọc Online Retail II (lần đầu khoảng 40 giây, sau đó đọc cache parquet trong thư mục tạm) và gọi World Bank API (không cần key).
- Chấm: `uv run pytest tracks/data-analyst/modules/05-data-collection/exercises/advanced/02-market-sizing-multi-source -q`

In [ ]:
import json
from datetime import datetime, timezone

import httpx
import pandas as pd

## 1. Nối tên nước giữa hai nguồn

`match_countries(names, wb_countries, aliases) -> pd.DataFrame`, một dòng cho mỗi tên trong `names` (giữ thứ tự), cột
`name`, `iso3`, `method`:

- tên có trong `aliases`: `iso3 = aliases[name]`, `method = "alias"`; nếu giá trị alias là `None` (cố ý không nối,
  như `European Community`) thì `iso3` thiếu và `method = "unmatched"`; alias trỏ tới mã không có trong `wb_countries`
  thì `ValueError` (lỗi gõ phải lộ ra ngay, không được âm thầm thành "không nối được");
- không có alias: so khớp với cột `name` của `wb_countries` sau khi bỏ khoảng trắng hai đầu và không phân biệt hoa
  thường; khớp thì `method = "exact"`;
- còn lại: `iso3` thiếu, `method = "unmatched"`.

`coverage(revenue, matched) -> dict`: `revenue` là Series doanh thu theo tên nước. Trả `matched_share` (phần doanh thu
của các tên có `iso3`) và `unmatched` (danh sách tên không nối được, sắp giảm dần theo doanh thu).

In [ ]:
def match_countries(names: list[str], wb_countries: pd.DataFrame, aliases: dict) -> pd.DataFrame:
    """name, iso3, method (alias | exact | unmatched) cho từng tên nước nội bộ."""
    # TODO
    raise NotImplementedError


def coverage(revenue: pd.Series, matched: pd.DataFrame) -> dict:
    """Phần doanh thu nối được và danh sách tên không nối được (giảm dần theo doanh thu)."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: doanh thu 2011 (tháng 1-11, đủ tháng) theo nước, theo định nghĩa chung của track
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

import os
import tempfile


def read_sheet(xlsx: Path, sheet: str) -> pd.DataFrame:
    # đọc một sheet, cache parquet trong thư mục tạm của hệ thống (khoá theo mtime/size của tệp gốc)
    stat = xlsx.stat()
    cache = (Path(tempfile.gettempdir()) / "da-cache"
             / f"online_retail_{sheet.replace(' ', '_')}_{stat.st_mtime_ns}_{stat.st_size}.parquet")
    if cache.exists():
        return pd.read_parquet(cache)
    df = pd.read_excel(xlsx, sheet_name=sheet, dtype={"Invoice": str, "StockCode": str, "Description": str})
    cache.parent.mkdir(parents=True, exist_ok=True)
    tmp = cache.with_name(f"{cache.name}.{os.getpid()}.tmp")
    df.to_parquet(tmp)
    os.replace(tmp, cache)
    return df

tx = read_sheet(ensure("online-retail-ii") / "online_retail_II.xlsx", "Year 2010-2011")
valid = tx[~tx.Invoice.str.startswith("C") & (tx.Quantity > 0) & (tx.Price > 0)
           & (tx.InvoiceDate >= "2011-01-01") & (tx.InvoiceDate < "2011-12-01")].copy()
valid["revenue"] = valid.Quantity * valid.Price
by_country = valid.groupby("Country").agg(revenue=("revenue", "sum"), customers=("Customer ID", "nunique"))
top_customer = valid.groupby(["Country", "Customer ID"]).revenue.sum().groupby("Country").max()
by_country["top_customer_share"] = (top_customer / by_country.revenue).fillna(0)
by_country = by_country.sort_values("revenue", ascending=False)
print(f"doanh thu 01-11/2011: {by_country.revenue.sum():,.0f} GBP, ngoài Anh: "
      f"{by_country.drop('United Kingdom').revenue.sum():,.0f} GBP")
by_country.head(12).round(2)

In [ ]:
# explore: danh sách nước của World Bank (bỏ nhóm "Aggregates") và nối tên
client = httpx.Client(base_url="https://api.worldbank.org/v2", timeout=30,
                      headers={"User-Agent": "da-learning-course/1.0"})
meta, countries = client.get("/country", params={"format": "json", "per_page": 400}).json()
assert len(countries) == int(meta["total"])
wb_countries = pd.DataFrame([{"iso3": c["id"], "name": c["name"], "region": c["region"]["value"]}
                             for c in countries if c["region"]["value"] != "Aggregates"])
print(len(wb_countries), "nước / vùng lãnh thổ")

names = list(by_country.index)
print("không khớp tên nguyên văn:", [n for n in names if n not in set(wb_countries.name)])

In [ ]:
# explore: alias do bạn quyết định, rồi kiểm phần doanh thu nối được
ALIASES = {
    "EIRE": "IRL", "USA": "USA", "RSA": "ZAF", "Czech Republic": "CZE", "Hong Kong": "HKG",
    "European Community": None,  # không phải một nước
    "Unspecified": None,
}
matched = match_countries(names, wb_countries, ALIASES)
print(matched.method.value_counts())
cov = coverage(by_country.revenue, matched)
print(f"doanh thu nối được: {cov['matched_share']:.4%}; không nối được: {cov['unmatched']}")

In [ ]:
# explore: dân số và GDP/người năm 2011 cho các nước đã nối, lưu JSON thô kèm metadata
RAW = ROOT / "data/raw_api/world-bank-retail"
RAW.mkdir(parents=True, exist_ok=True)
iso = sorted(matched.iso3.dropna().unique())
wb = {}
for indicator in ["SP.POP.TOTL", "NY.GDP.PCAP.CD"]:
    meta, data = client.get(f"/country/{';'.join(iso)}/indicator/{indicator}",
                            params={"format": "json", "date": "2011", "per_page": 1000}).json()
    assert meta["pages"] == 1 and len(data) == int(meta["total"])
    (RAW / f"{indicator}_2011.json").write_text(json.dumps({
        "fetched_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "lastupdated": meta["lastupdated"], "records": data}))
    wb[indicator] = {r["countryiso3code"]: r["value"] for r in data}
    print(indicator, len(data), "bản ghi, lastupdated", meta["lastupdated"])

In [ ]:
# explore: bảng xếp hạng châu Âu (trừ Anh) - doanh thu theo quy mô thị trường và độ tập trung khách
europe = set(wb_countries.loc[wb_countries.region == "Europe & Central Asia", "iso3"])
table = (by_country.join(matched.set_index("name"))
         .query("iso3 in @europe and iso3 != 'GBR'")
         .assign(population_m=lambda d: d.iso3.map(wb["SP.POP.TOTL"]) / 1e6,
                 gdp_per_capita=lambda d: d.iso3.map(wb["NY.GDP.PCAP.CD"]),
                 revenue_per_m_pop=lambda d: d.revenue / d.population_m,
                 revenue_per_customer=lambda d: d.revenue / d.customers))
cols = ["revenue", "customers", "top_customer_share", "population_m", "gdp_per_capita", "revenue_per_m_pop",
        "revenue_per_customer"]
print(table[cols].sort_values("revenue", ascending=False).head(12).round(2).to_string())

In [ ]:
# explore: biểu đồ - doanh thu so với quy mô thị trường, cỡ điểm = số khách
import matplotlib.pyplot as plt

plot = table[table.revenue > 10_000]
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(plot.population_m * plot.gdp_per_capita / 1e3, plot.revenue / 1e3, s=plot.customers * 3 + 10,
           alpha=0.6, color="#4c78a8")
for name, row in plot.iterrows():
    ax.annotate(name, (row.population_m * row.gdp_per_capita / 1e3, row.revenue / 1e3), fontsize=8,
                xytext=(4, 2), textcoords="offset points")
ax.set_xscale("log")
ax.set_xlabel("quy mô kinh tế: dân số × GDP/người (tỉ USD, log)")
ax.set_ylabel("doanh thu 01-11/2011 (nghìn GBP)")
ax.set_title("Đức, Pháp: kinh tế lớn, nhiều khách; Ireland, Hà Lan: doanh thu dồn vào vài khách", loc="left",
             fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 2. Viết memo

Mở `starter/MEMO.md`, viết cho Sales Director (300–1 000 từ, 6 mục theo thứ tự có sẵn). Memo tốt nói kết luận trước,
đưa ít con số nhưng đúng, và nói rõ số nào **không** đáng tin (doanh thu một nước đến từ một hai khách sỉ không phải
"thị trường"). Mục "Nguồn dữ liệu" ghi đủ để người khác lấy lại đúng số: bộ dữ liệu, sheet, khoảng thời gian, định
nghĩa doanh thu, API, chỉ số, `lastupdated`, ngày tải.